In [1]:
import pandas as pd
df = pd.read_csv(r"C:\LODAT\test_assets\large_data.csv", index_col=0)
df.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
0,0.092692,2.055946,7.821476,2000,HH,-4.088469
1,-0.049396,2.001348,-3.037385,2000,HH,-5.566499
2,-0.083841,3.754457,6.129944,2000,HH,-8.347156
3,-0.070949,3.089924,6.610768,2000,HH,-6.338199
4,0.130640,3.371985,-8.729374,2000,HH,3.344893


In [2]:
def slice_data(dataframe, frequency, polarization, look_center, depr_center, bin_size):
    look_width, depr_width = bin_size

    look_min = look_center - look_width/2
    look_max = look_center + look_width/2

    depr_min = depr_center - depr_width/2
    depr_max = depr_center + depr_width/2

    vector_mask = (dataframe.Frequency == float(frequency)) & (dataframe.Polarization == polarization)
    depr_mask = (dataframe.Depression >= depr_min) & (dataframe.Depression < depr_max)
    look_mask = (dataframe.Look >= look_min) & (dataframe.Look < look_max)
    mask = vector_mask & depr_mask & look_mask
    return dataframe[mask]

# Generate some fake data

In [3]:
from itertools import product
import numpy as np
freqs = df.Frequency.unique()[:5]
pols = df.Polarization.unique()
looks = np.arange(0, 1)
deprs = [2.5]
bin_size = [(1, 5)]
params = list(product(freqs, pols, looks, deprs, bin_size))

In [4]:
iterations = len(params)
print(f'This will be the number of loops: {iterations}')

This will be the number of loops: 10


# Timing

Traditional Looping

In [5]:
import time
import pandas as pd

In [6]:
t0 = time.time()
dataframes1 = []
for param in params:
    dataframes1.append(
        slice_data(df, *param)
    )
result1 = pd.concat(dataframes1)
t1 = time.time()
print(t1-t0)
result1.shape

2.4735307693481445


(1011, 6)

Multi-threading

In [7]:
from concurrent.futures import ThreadPoolExecutor

In [8]:
t0 = time.time()
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(slice_data, df, *param) for param in params]
    dataframes2 = [future.result() for future in futures]
result2 = pd.concat(dataframes2)
t1 = time.time()
print(t1-t0)
result2.shape

1.7088353633880615


(1011, 6)

Multi-processing

In [9]:
from concurrent.futures import ProcessPoolExecutor

In [11]:
if __name__ == '__main__':
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(slice_data, df, *param) for param in params]
        dataframes2 = [future.result() for future in futures]
    result2 = pd.concat(dataframes2)
    t1 = time.time()
    print(t1-t0)
    result2.shape

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

# Using Groupby

In [13]:
df.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
0,0.092692,2.055946,7.821476,2000,HH,-4.088469
1,-0.049396,2.001348,-3.037385,2000,HH,-5.566499
2,-0.083841,3.754457,6.129944,2000,HH,-8.347156
3,-0.070949,3.089924,6.610768,2000,HH,-6.338199
4,0.130640,3.371985,-8.729374,2000,HH,3.344893


In [15]:
grouped = df.groupby(['Frequency', 'Polarization'])

In [17]:
grouped[grouped.Look > 0]

TypeError: '>' not supported between instances of 'SeriesGroupBy' and 'int'

In [18]:
for name, data in grouped:
    print(name)
    print(data.head())

(2000, 'HH')
       Look  Depression     Twist  Frequency Polarization       RCS
0  0.092692    2.055946  7.821476       2000           HH -4.088469
1 -0.049396    2.001348 -3.037385       2000           HH -5.566499
2 -0.083841    3.754457  6.129944       2000           HH -8.347156
3 -0.070949    3.089924  6.610768       2000           HH -6.338199
4  0.130640    3.371985 -8.729374       2000           HH  3.344893
(2000, 'VV')
       Look  Depression     Twist  Frequency Polarization        RCS
0 -0.014236    2.884426 -4.245023       2000           VV   1.653626
1  0.006038    2.071937 -2.910947       2000           VV  -4.552857
2 -0.013673    2.551222 -3.036474       2000           VV  -2.594657
3  0.105548    2.426242 -6.977290       2000           VV  12.168184
4 -0.035004    2.272887 -2.780725       2000           VV   2.099081
(3000, 'HH')
       Look  Depression     Twist  Frequency Polarization       RCS
0 -0.069099    2.606486 -0.686578       3000           HH  2.627716
1 -

In [19]:
grouped['2000']['HH']

KeyError: 'Column not found: 2000'

In [20]:
grouped.get_group((2000, 'HH'))

,Look,Depression,Twist,Frequency,Polarization,RCS
0,0.092692,2.055946,7.821476,2000,HH,-4.088469
1,-0.049396,2.001348,-3.037385,2000,HH,-5.566499
2,-0.083841,3.754457,6.129944,2000,HH,-8.347156
3,-0.070949,3.089924,6.610768,2000,HH,-6.338199
4,0.130640,3.371985,-8.729374,2000,HH,3.344893
...,...,...,...,...,...,...
71995,359.950117,-8.791742,1.533591,2000,HH,-0.902458
71996,359.953025,-6.829760,3.506636,2000,HH,-0.604377
71997,360.017066,-7.737004,-13.999400,2000,HH,-1.625806
71998,359.827083,-8.128937,2.075474,2000,HH,-1.470557


In [ ]:
# TODO maybe group by vector and then pass to multiple processes to slice